# Cuaderno 08. OCR de mecanografiados: modelos de visión y lenguaje contra OCR clásicos

## 0. Instalar dependencias (solo Colab)

In [ ]:
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    subprocess.run("apt-get -qq install -y tesseract-ocr > /dev/null", shell=True, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=5.5.4", "accelerate", "bitsandbytes",
                    "python-doctr", "pytesseract", "jiwer"], check=True)
    print("Instalados Tesseract, transformers, docTR y jiwer")
else:
    print("Fuera de Colab no se instala nada: ver los paquetes en cuadernos/README.md.")

## 1. Funciones de texto

`limpiar()` quita las marcas de formato (XML, HTML, Markdown), une las palabras cortadas con guion al final de línea y reduce los espacios y saltos de línea a uno.

In [ ]:
import html
import re


def limpiar(texto):
    """Deja solo el texto: sin marcas XML, HTML o Markdown, con las palabras cortadas
    por guion al final de línea unidas, y con un solo espacio entre palabras."""
    texto = re.sub(r"<Metadata>.*?</Metadata>", "", texto, flags=re.S)  # ficha que Churro agrega antes del texto
    texto = re.sub(r"</(Line|p|div|li|h\d)>|<br\s*/?>|<Gap[^>]*/>", " ", texto)  # fin de línea o de párrafo
    texto = re.sub(r"<[^>]+>", "", texto)
    texto = html.unescape(texto).replace("**", "")
    texto = re.sub(r"^#+\s*", "", texto, flags=re.M)
    texto = re.sub(r"(\w)-\s+(?=[a-záéíóúüñ])", r"\1", texto)  # "pala- bra" -> "palabra"
    return re.sub(r"\s+", " ", texto).strip()

## 2. Cargar los párrafos

Lee cinco párrafos del [OCR Berrutti Dataset v3](https://github.com/camilomarino/ocr_berrutti_dataset) (Universidad de la República / Centre Borelli, ENS Paris-Saclay; licencia CC BY-NC-SA 4.0) guardados en `datos_ocr/mecanografiado/`: el recorte (`.png`) y su transcripción de referencia (`.txt`). En Colab los descarga del repositorio del taller.

In [ ]:
import urllib.request
from pathlib import Path

import pandas as pd
from PIL import Image

REPO_RAW = "https://raw.githubusercontent.com/agusnieto77/sicss-uruguay-2026/main/cuadernos/datos_ocr/mecanografiado/"
CARPETA = Path("datos_ocr/mecanografiado")  # fuera de Colab se leen del repositorio clonado
PAGINAS = [
    "high_r0809_1437_p01",
    "high_r0809_1437_p04",
    "high_r0126_0309_p02",
    "high_r0813_0921_p01",
    "high_r0814_0185_p01",
]


def leer(nombre):
    if not (CARPETA / nombre).exists():
        CARPETA.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + nombre, CARPETA / nombre)
    return CARPETA / nombre


paginas = []
for id_ in PAGINAS:
    imagen = Image.open(leer(id_ + ".png")).convert("RGB")
    referencia = leer(id_ + ".txt").read_text(encoding="utf-8")
    paginas.append({"id": id_, "imagen": imagen, "referencia": limpiar(referencia)})
print(len(paginas), "párrafos")

In [ ]:
from IPython.display import display

for pagina in paginas:
    miniatura = pagina["imagen"].copy()
    miniatura.thumbnail((900, 900))
    print(pagina["id"], "-", len(pagina["referencia"]), "caracteres en la referencia")
    display(miniatura)

## 3. OCR clásicos: docTR y Tesseract

Carga [docTR](https://github.com/mindee/doctr) y [Tesseract](https://github.com/tesseract-ocr/tesseract) y define una función de transcripción para cada uno. Tesseract usa el modelo de español `tessdata_best` y `--psm 6` (la imagen como un solo bloque de texto).

In [ ]:
import time
import urllib.request
from pathlib import Path

import numpy as np
import pytesseract
import torch
from doctr.models import ocr_predictor

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

# docTR: detección db_resnet50 + reconocimiento crnn_vgg16_bn
modelo_doctr = ocr_predictor(pretrained=True).to(DISPOSITIVO)


def transcribir_doctr(imagen):
    pagina = modelo_doctr([np.array(imagen)]).export()["pages"][0]
    lineas = [" ".join(palabra["value"] for palabra in linea["words"])
              for bloque in pagina["blocks"] for linea in bloque["lines"]]
    return "\n".join(lineas)


# Tesseract con el modelo de español más preciso (tessdata_best) y --psm 6: un bloque de texto uniforme.
TESSDATA = Path("tessdata")
if not (TESSDATA / "spa.traineddata").exists():
    TESSDATA.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://github.com/tesseract-ocr/tessdata_best/raw/main/spa.traineddata",
                               TESSDATA / "spa.traineddata")


def transcribir_tesseract(imagen):
    return pytesseract.image_to_string(imagen, lang="spa", config=f"--tessdata-dir {TESSDATA} --psm 6")


print("Tesseract", pytesseract.get_tesseract_version(), "- docTR en", DISPOSITIVO)

## 4. Modelos de visión y lenguaje

`cargar()` descarga un modelo y lo pone en la GPU; con `cuantizar=True` lo carga en 4 bits. `transcribir()` arma el mensaje (imagen, instrucción de sistema e instrucción de usuario, según el modelo), genera el texto y lo devuelve crudo.

In [ ]:
import gc

from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

# bfloat16 en GPU modernas (L4, A100); float16 en la T4, que no tiene bfloat16 nativo; float32 sin GPU.
if not torch.cuda.is_available():
    TIPO = torch.float32
elif torch.cuda.get_device_capability()[0] >= 8:
    TIPO = torch.bfloat16
else:
    TIPO = torch.float16


def cargar(repo, cuantizar=False, codigo_propio=False, **_):
    opciones = {"dtype": TIPO, "device_map": DISPOSITIVO, "trust_remote_code": codigo_propio}
    if cuantizar:  # 4 bits (NF4): el modelo ocupa la cuarta parte de memoria
        opciones["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                             bnb_4bit_compute_dtype=TIPO)
    clase = AutoModelForCausalLM if codigo_propio else AutoModelForImageTextToText
    return clase.from_pretrained(repo, **opciones), AutoProcessor.from_pretrained(repo, trust_remote_code=codigo_propio)


def transcribir(modelo, procesador, imagen, sistema=None, instruccion=None, muestreo=None, plantilla=None, **_):
    contenido = [{"type": "image", "image": imagen}]
    if instruccion:
        contenido.append({"type": "text", "text": instruccion})
    mensajes = [{"role": "user", "content": contenido}]
    if sistema:
        mensajes.insert(0, {"role": "system", "content": [{"type": "text", "text": sistema}]})
    entradas = procesador.apply_chat_template(mensajes, add_generation_prompt=True, tokenize=True, return_dict=True,
                                              return_tensors="pt", **(plantilla or {})).to(modelo.device)
    # Todos estos modelos derivan de Qwen y terminan cada respuesta con <|im_end|>.
    fin = procesador.tokenizer.convert_tokens_to_ids("<|im_end|>")
    salida = modelo.generate(**entradas, max_new_tokens=MAX_TOKENS, eos_token_id=fin, **(muestreo or {"do_sample": False}))
    return procesador.decode(salida[0, entradas["input_ids"].shape[1]:], skip_special_tokens=True)

`MODELOS` define qué modelos se corren, con la instrucción y el muestreo de cada uno. Borrá una entrada para no correr ese modelo. Qwen3-VL-8B se carga en 4 bits para que entre en una T4. `MAX_TOKENS` limita el texto generado por página.

In [ ]:
MAX_TOKENS = 2048  # tope de texto generado por página; corta los bucles de repetición

# Instrucción de nuestra evaluación para modelos de propósito general.
PROMPT_MECANOGRAFIADO = (
    "OCR this scanned paragraph from a historical typewritten Uruguayan document. Return only "
    "plain text. Preserve original line breaks, spelling, accents, capitalization, punctuation, "
    "abbreviations, dates, numbers, and hyphenation. Do not explain. Do not summarize. Do not "
    "use markdown. Do not correct or expand abbreviations. Do not invent missing text. Use "
    "[ilegible] only for unreadable text."
)
# Final de la instrucción oficial de Chandra (repositorio datalab-to/chandra).
CHANDRA_FIN = """Only use these tags ['math', 'br', 'i', 'b', 'u', 'del', 'sup', 'sub', 'table', 'tr', 'td', 'p', 'th', 'div', 'pre', 'h1', 'h2', 'h3', 'h4', 'h5', 'ul', 'ol', 'li', 'input', 'a', 'span', 'img', 'hr', 'tbody', 'small', 'caption', 'strong', 'thead', 'big', 'code', 'chem'], and these attributes ['class', 'colspan', 'rowspan', 'display', 'checked', 'type', 'border', 'value', 'style', 'href', 'alt', 'align', 'data-bbox', 'data-label'].

Guidelines:
* Inline math: Surround math with <math>...</math> tags. Math expressions should be rendered in KaTeX-compatible LaTeX. Use display for block math.
* Tables: Use colspan and rowspan attributes to match table structure.
* Formatting: Maintain consistent formatting with the image, including spacing, indentation, subscripts/superscripts, and special characters.
* Images: Include a description of any images in the alt attribute of an <img> tag. Do not fill out the src property. Describe in detail inside the div tag. Also convert charts to high fidelity data, and convert diagrams to mermaid.
* Forms: Mark checkboxes and radio buttons properly.
* Text: join lines together properly into paragraphs using <p>...</p> tags.  Use <br> tags for line breaks within paragraphs, but only when absolutely necessary to maintain meaning.
* Chemistry: Use <chem>...</chem> tags for chemical formulas with reactive SMILES.
* Lists: Preserve indents and proper list markers.
* Use the simplest possible HTML structure that accurately represents the content of the block.
* Make sure the text is accurate and easy for a human to read and interpret.  Reading order should be correct and natural."""

# Borrá una entrada para no correr ese modelo. Se cargan de a uno: en una T4 no entran dos juntos.
MODELOS = {
    "LightOnOCR-3-4B": {"repo": "lightonai/LightOnOCR-3-4B",  # sin instrucción: solo transcribe
                        "plantilla": {"enable_thinking": False},
                        "muestreo": {"do_sample": True, "temperature": 0.2, "top_p": 0.9}},
    "Qwen3-VL-8B (4 bits)": {"repo": "Qwen/Qwen3-VL-8B-Instruct", "instruccion": PROMPT_MECANOGRAFIADO, "cuantizar": True},
    "Chandra OCR 2": {"repo": "datalab-to/chandra-ocr-2", "instruccion": "OCR this image to HTML.\n\n" + CHANDRA_FIN,
                      "plantilla": {"enable_thinking": False}},
}

## 5. Transcribir

Pasa cada página por los cinco sistemas y mide los segundos que tarda cada uno. Los modelos de visión y lenguaje se cargan de a uno y se liberan antes del siguiente.

In [ ]:
filas = []


def medir(sistema, transcribir_pagina):
    for pagina in paginas:
        inicio = time.perf_counter()
        texto = transcribir_pagina(pagina["imagen"])
        segundos = time.perf_counter() - inicio
        filas.append({"pagina": pagina["id"], "sistema": sistema, "segundos": round(segundos, 1), "texto": texto})
        print(f"{sistema:>22}  {segundos:6.1f} s  {pagina['id']}")


medir("docTR", transcribir_doctr)
medir("Tesseract", transcribir_tesseract)
for nombre, opciones in MODELOS.items():
    modelo, procesador = cargar(**opciones)
    torch.manual_seed(0)
    medir(nombre, lambda imagen: transcribir(modelo, procesador, imagen, **opciones))
    del modelo, procesador  # libera la GPU antes de cargar el modelo siguiente
    gc.collect()
    torch.cuda.empty_cache()

transcripciones = pd.DataFrame(filas)

## 6. CER y WER

Calcula el CER y el WER de cada sistema en cada página contra la referencia, después de aplicar `limpiar()` a las dos. **CER**: ediciones de un carácter (sustituir, borrar o agregar) sobre la cantidad de caracteres de la referencia; **WER**: lo mismo en palabras. 0 % es una transcripción perfecta. El resumen muestra la mediana por sistema y el **CER total** (todos los errores sobre todos los caracteres).

In [ ]:
import jiwer

referencias = {p["id"]: p["referencia"] for p in paginas}
transcripciones["limpio"] = transcripciones["texto"].map(limpiar)
transcripciones["CER"] = [100 * jiwer.cer(referencias[p], t) for p, t in zip(transcripciones["pagina"], transcripciones["limpio"])]
transcripciones["WER"] = [100 * jiwer.wer(referencias[p], t) for p, t in zip(transcripciones["pagina"], transcripciones["limpio"])]

columnas = ["CER", "WER"]
orden = list(dict.fromkeys(transcripciones["sistema"]))
for columna in columnas:
    print(columna, "por página (%):")
    display(transcripciones.pivot(index="pagina", columns="sistema", values=columna)[orden].round(2))

resumen = transcripciones.groupby("sistema")[columnas + ["segundos"]].median().round(2).add_suffix(" (mediana)")
for sistema, grupo in transcripciones.groupby("sistema"):
    resumen.loc[sistema, "CER total"] = round(100 * jiwer.cer([referencias[p] for p in grupo["pagina"]], list(grupo["limpio"])), 2)
print("Resumen (%; segundos por página):")
display(resumen.sort_values("CER total"))

## 7. Comparar una página

Muestra la imagen, la referencia y las transcripciones limpias. Cambiá `PAGINA` para ver otra.

In [ ]:
PAGINA = 0  # número de página (0 es la primera)

pagina = paginas[PAGINA]
display(pagina["imagen"])
print("REFERENCIA\n" + pagina["referencia"] + "\n")
for _, fila in transcripciones[transcripciones["pagina"] == pagina["id"]].iterrows():
    print(f"{fila['sistema'].upper()} (CER {fila['CER']:.2f} %)\n{fila['limpio']}\n")

## 8. Guardar los resultados

Guarda en `resultados/08_ocr_mecanografiado/` el texto crudo de cada sistema en cada página, las métricas por página y el resumen. En Colab los descarga como `.zip`.

In [ ]:
import shutil

SALIDA = Path("resultados") / "08_ocr_mecanografiado"
(SALIDA / "textos").mkdir(parents=True, exist_ok=True)
for _, fila in transcripciones.iterrows():
    nombre = re.sub(r"[^\w.-]+", "_", fila["pagina"] + "__" + fila["sistema"]) + ".txt"
    (SALIDA / "textos" / nombre).write_text(fila["texto"], encoding="utf-8")
transcripciones.drop(columns="texto").to_csv(SALIDA / "metricas_por_pagina.csv", index=False)
resumen.to_csv(SALIDA / "resumen.csv")
print("Resultados en", SALIDA)

if EN_COLAB:
    from google.colab import files
    files.download(shutil.make_archive(str(SALIDA), "zip", SALIDA))